# Assignment 4 – Portugal | Part 3: Visualizations

**Requires outputs from `EAnalysis.ipynb` and `PyPSA.ipynb`.**

Covers all *Outputs of interest* from the assignment:
1. Total system cost split by technology
2. Capacity built per technology
3. Electricity mix
4. CO₂ shadow price
5. Price duration curves
6. Average electricity prices per region
7. Rate of curtailment
8. Storage filling levels
9. System operation (example week)
10. Sensitivity analysis

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt, cartopy.crs as ccrs, cartopy
import pypsa

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

COLORS = {
    'solar':    '#f4a261',  # orange
    'onwind':   '#457b9d',  # mid blue
    'offwind':  '#1d3557',  # dark navy
    'hydro':    '#3a86ff',  # water blue
    'gas':      '#a0522d',  # brown
    'coal':     '#343a40',  # dark grey
    'biomass':  '#2d6a4f',  # forest green
    'waste':    '#e9c46a',  # sand yellow
    'battery':  '#2a9d8f',  # teal
    'H2':       '#e76f51',  # coral red
    'AC':       '#6c757d',  # grey
}
print('Setup complete.')

## Load data

In [ ]:
n_free = pypsa.Network('network_free.nc')
n_zero = pypsa.Network('network_zero.nc')
regions = gpd.read_file('regions.gpkg').set_index('nuts2')
regions['representative_point'] = regions.geometry.representative_point()
df_sens = pd.read_csv('sensitivity_results.csv', index_col=0)
print('n_free generators:', len(n_free.generators))
print('n_zero generators:', len(n_zero.generators))
df_sens

---
## Fig 1 · Total system cost split by technology

In [ ]:
def cost_by_carrier(net):
    rows = []
    for name, g in net.generators.iterrows():
        dispatch = net.generators_t.p[name].multiply(net.snapshot_weightings.generators).sum()
        rows.append({'carrier': g.carrier,
                     'capital':  g.capital_cost * g.p_nom_opt,
                     'marginal': g.marginal_cost * dispatch})
    for name, s in net.storage_units.iterrows():
        rows.append({'carrier': s.carrier,
                     'capital':  s.capital_cost * s.p_nom_opt, 'marginal': 0})
    for name, lk in net.links.iterrows():
        rows.append({'carrier': 'AC', 'capital': lk.capital_cost * lk.p_nom_opt, 'marginal': 0})
    df = pd.DataFrame(rows).groupby('carrier')[['capital', 'marginal']].sum() / 1e9
    return df[df.sum(axis=1) > 0.001]

cost_free = cost_by_carrier(n_free)
cost_zero = cost_by_carrier(n_zero)
all_c = sorted(set(cost_free.index) | set(cost_zero.index))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title in [(axes[0], cost_free, 'No CO₂ limit'), (axes[1], cost_zero, 'Net-zero CO₂')]:
    df = df.reindex(all_c, fill_value=0)
    for i, c in enumerate(all_c):
        color = COLORS.get(c, '#adb5bd')
        ax.bar(i, df.at[c, 'capital'],  color=color, label=c if ax is axes[0] else '_')
        ax.bar(i, df.at[c, 'marginal'], bottom=df.at[c, 'capital'], color=color, alpha=0.4, hatch='//')
    ax.set_xticks(range(len(all_c))); ax.set_xticklabels(all_c, rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('bn EUR/year')
    ax.set_title(f'System Cost – {title}\n(total: {df.sum().sum():.2f} bn EUR/a)', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
axes[0].legend(bbox_to_anchor=(1, 1), fontsize=8, title='solid=capital, hatch=marginal')
plt.tight_layout()
plt.savefig('fig1_system_cost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 2 · Installed capacity per technology

In [ ]:
def cap_by_carrier(net):
    gen  = net.generators.groupby('carrier')['p_nom_opt'].sum() / 1e3
    stor = net.storage_units.groupby('carrier')['p_nom_opt'].sum() / 1e3
    tx   = pd.Series({'transmission': net.links['p_nom_opt'].sum() / 1e3})
    return pd.concat([gen, stor, tx])

cap = pd.concat({'No CO₂ limit': cap_by_carrier(n_free),
                  'Net-zero':      cap_by_carrier(n_zero)}, axis=1).fillna(0)
cap = cap[cap.sum(axis=1) > 0.01].sort_values('Net-zero', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(cap)); w = 0.35
colors = [COLORS.get(c, '#adb5bd') for c in cap.index]
ax.barh(x - w/2, cap['No CO₂ limit'], w, color=colors, alpha=0.6, label='No CO₂ limit')
ax.barh(x + w/2, cap['Net-zero'],      w, color=colors,            label='Net-zero')
ax.set_yticks(x); ax.set_yticklabels(cap.index)
ax.set_xlabel('Installed capacity [GW]')
ax.set_title('Optimal Installed Capacity', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_capacity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 3 · Electricity mix (annual generation)

In [ ]:
def gen_mix(net):
    return (net.generators_t.p
               .multiply(net.snapshot_weightings.generators, axis=0)
               .groupby(net.generators.carrier, axis=1).sum()
               .sum() / 1e6)  # TWh

mix = pd.concat({'No CO₂ limit': gen_mix(n_free), 'Net-zero': gen_mix(n_zero)}, axis=1).fillna(0)
mix = mix[mix.sum(axis=1) > 0.01]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in [(axes[0], 'No CO₂ limit'), (axes[1], 'Net-zero')]:
    data   = mix[col][mix[col] > 0].sort_values(ascending=False)
    colors = [COLORS.get(c, '#adb5bd') for c in data.index]
    ax.pie(data.values, labels=data.index, colors=colors,
           autopct=lambda p: f'{p:.1f}%' if p > 2 else '', startangle=90, pctdistance=0.8)
    ax.set_title(f'{col}\nTotal: {data.sum():.1f} TWh/a', fontweight='bold')
plt.suptitle('Annual Electricity Generation Mix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_generation_mix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 4 · CO₂ shadow price

In [ ]:
if 'co2_limit' in n_zero.global_constraints.index:
    co2_price = -n_zero.global_constraints.at['co2_limit', 'mu']
    print(f'CO2 shadow price: {co2_price:.1f} EUR/tCO2')
else:
    co2_price = None
    print('CO2 constraint mu not found.')

fig, ax = plt.subplots(figsize=(5, 4))
if co2_price:
    ax.bar(['Portugal\nNet-zero'], [co2_price], color='#e63946', width=0.4)
    ax.axhline(100, color='grey', linestyle='--', label='EU ETS ~100 EUR/t')
    ax.set_ylabel('EUR / tCO₂')
    ax.set_title('CO₂ Shadow Price', fontweight='bold')
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig4_co2_price.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 5 · Price duration curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, net, title in [(axes[0], n_free, 'No CO₂ limit'), (axes[1], n_zero, 'Net-zero CO₂')]:
    for bus in net.buses_t.marginal_price.columns:
        sorted_p = net.buses_t.marginal_price[bus].sort_values(ascending=False).values
        hours = np.arange(len(sorted_p)) * 3
        ax.plot(hours, sorted_p, label=bus, linewidth=1.5, alpha=0.85)
    ax.set_xlabel('Hours per year'); ax.set_ylabel('Marginal price [EUR/MWh]')
    ax.set_title(f'Price Duration Curve – {title}', fontweight='bold')
    ax.legend(fontsize=8); ax.set_xlim(0, 8760); ax.set_ylim(bottom=0); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig5_price_duration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 6 · Average electricity prices per region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, net, title in [(axes[0], n_free, 'No CO₂ limit'), (axes[1], n_zero, 'Net-zero CO₂')]:
    avg = net.buses_t.marginal_price.mean()
    merged = regions.copy(); merged['price'] = avg
    merged.plot(column='price', ax=ax, cmap='YlOrRd', legend=True,
                legend_kwds={'label': 'EUR/MWh', 'shrink': 0.6})
    for region, row in regions.iterrows():
        pt = row['representative_point']
        ax.annotate(f"{region}\n{avg.get(region, 0):.1f}",
                    (pt.x, pt.y), ha='center', fontsize=8, fontweight='bold')
    ax.set_title(f'Avg Electricity Price – {title}', fontweight='bold'); ax.set_axis_off()
plt.tight_layout()
plt.savefig('fig6_avg_prices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 7 · Curtailment

In [ ]:
def curtailment(net):
    rows = []
    for name, g in net.generators.iterrows():
        if g.carrier not in ['solar', 'onwind', 'offwind']: continue
        w     = net.snapshot_weightings.generators
        avail = (net.generators_t.p_max_pu[name] * g.p_nom_opt * w).sum()
        actual = (net.generators_t.p[name] * w).sum()
        rows.append({'carrier': g.carrier, 'avail': avail, 'curtailed': avail - actual})
    df = pd.DataFrame(rows).groupby('carrier')[['avail','curtailed']].sum()
    df['rate_%'] = (df['curtailed'] / df['avail'].replace(0, np.nan) * 100).fillna(0)
    return df

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, net, title in [(axes[0], n_free, 'No CO₂ limit'), (axes[1], n_zero, 'Net-zero CO₂')]:
    df = curtailment(net)
    bars = ax.bar(df.index, df['rate_%'],
                  color=[COLORS.get(c, '#adb5bd') for c in df.index])
    ax.bar_label(bars, fmt='%.1f%%', padding=3)
    ax.set_ylabel('Curtailment [%]')
    ax.set_title(f'Renewable Curtailment – {title}', fontweight='bold')
    ax.set_ylim(0, max(df['rate_%'].max() * 1.3, 5))
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig7_curtailment.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 8 · Storage filling levels

In [ ]:
soc = n_zero.storage_units_t.state_of_charge
batt_soc = soc[[c for c in soc.columns if 'battery' in c]].sum(axis=1)
h2_soc   = soc[[c for c in soc.columns if 'H2' in c]].sum(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, series, label, color in [
    (axes[0], batt_soc.resample('ME').mean() / 1e3, 'Battery [GWh]', COLORS['battery']),
    (axes[1], h2_soc.resample('ME').mean()   / 1e3, 'Hydrogen [GWh]', COLORS['H2']),
]:
    ax.fill_between(series.index, series.values, alpha=0.7, color=color)
    ax.set_ylabel(label); ax.grid(True, alpha=0.3)
    ax.set_title(f'{label} – Monthly Average SoC (Net-zero)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_storage_levels.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 9 · System operation — example weeks

In [ ]:
def plot_week(net, start, axes):
    snap = net.snapshots
    week = snap[(snap >= start) & (snap < pd.Timestamp(start) + pd.Timedelta('7D'))]
    gen  = net.generators_t.p.loc[week].groupby(net.generators.carrier, axis=1).sum() / 1e3
    stor = net.storage_units_t.p.loc[week].groupby(net.storage_units.carrier, axis=1).sum() / 1e3
    load = net.loads_t.p_set.loc[week].sum(axis=1) / 1e3

    ax = axes[0]
    bottom = pd.Series(0., index=week)
    for c in gen.columns:
        if gen[c].abs().max() < 0.01: continue
        ax.fill_between(week, bottom, bottom + gen[c], label=c,
                        color=COLORS.get(c, '#adb5bd'), alpha=0.9, step='post')
        bottom += gen[c]
    ax.plot(week, load, color='black', linewidth=2, label='Load', zorder=10)
    ax.set_ylabel('Power [GW]'); ax.legend(fontsize=7, ncol=3); ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    for c in stor.columns:
        ax2.plot(week, stor[c], label=c, color=COLORS.get(c, '#adb5bd'), linewidth=1.5)
    ax2.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax2.set_ylabel('Storage [GW]\n(+out / -in)'); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

fig, axes = plt.subplots(4, 1, figsize=(14, 14))
axes[0].set_title('Summer week (July 2013) – Net-zero', fontweight='bold')
plot_week(n_zero, '2013-07-01', axes[0:2])
axes[2].set_title('Winter week (January 2013) – Net-zero', fontweight='bold')
plot_week(n_zero, '2013-01-07', axes[2:4])
plt.tight_layout()
plt.savefig('fig9_dispatch_weeks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fig 10 · Sensitivity analysis

In [ ]:
x = df_sens.index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cost
axes[0].plot(x, df_sens['total_cost_bn_eur'], marker='o', linewidth=2, color='steelblue')
for xi, yi in zip(x, df_sens['total_cost_bn_eur']):
    axes[0].annotate(f'{yi:.2f}', (xi, yi), xytext=(0, 8), textcoords='offset points',
                     ha='center', fontsize=9)
axes[0].set_title('Total System Cost', fontweight='bold')
axes[0].set_ylabel('bn EUR/year'); axes[0].set_xlabel('Solar PV capital cost')
axes[0].grid(True, alpha=0.3)

# Capacity stacked bar
bottom = np.zeros(len(x))
for col, label, color in [
    ('solar_GW',   'Solar PV',      COLORS['solar']),
    ('onwind_GW',  'Onshore wind',  COLORS['onwind']),
    ('offwind_GW', 'Offshore wind', COLORS['offwind']),
    ('OCGT_GW',    'OCGT',          COLORS['gas']),
]:
    vals = df_sens[col].values
    axes[1].bar(x, vals, bottom=bottom, label=label, color=color, alpha=0.9)
    bottom += vals
axes[1].set_title('Installed Capacity', fontweight='bold')
axes[1].set_ylabel('GW'); axes[1].set_xlabel('Solar PV capital cost')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3, axis='y')

# Storage
axes[2].plot(x, df_sens['battery_GW'], marker='s', label='Battery',   color=COLORS['battery'], linewidth=2)
axes[2].plot(x, df_sens['H2_GW'],      marker='^', label='Hydrogen',  color=COLORS['H2'],      linewidth=2)
axes[2].plot(x, df_sens['transmission_GW'], marker='D', label='Tx',   color=COLORS['AC'],      linewidth=2)
axes[2].set_title('Storage & Transmission', fontweight='bold')
axes[2].set_ylabel('GW'); axes[2].set_xlabel('Solar PV capital cost')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle('Sensitivity: Solar PV Capital Cost (Net-zero CO₂)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig10_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()